# LangSmith: Tracing, Datasets & Experiments

In [2]:
%pip install langsmith openai openevals datasets python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads LANGSMITH_API_KEY, OPENAI_API_KEY, ANTHROPIC_API_KEY from .env

# LangSmith EU endpoint — set here so it takes effect before the client initialises
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "week7-evaluation"

In [4]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tracing

`@traceable` logs every call to LangSmith automatically. `wrap_openai` adds tracing to the OpenAI client.

In [5]:
@traceable(name="simple-qa")
def answer_question(question: str) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer concisely."},
            {"role": "user",   "content": question},
        ],
    )
    return response.choices[0].message.content.strip()

# This call will appear in your LangSmith project under 'week7-evaluation'
answer = answer_question("What is the speed of light?")
print(answer)

The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers per second or 186,282 miles per second).


## 2. Benchmark dataset from HuggingFace (HLE)

[Humanity's Last Exam](https://huggingface.co/datasets/cais/hle) — expert-level questions across many domains. We use text-only multiple-choice examples.

In [6]:
# Load text-only, multiple-choice examples
ds = load_dataset("cais/hle", split="test", streaming=True)

hle_examples = []
for sample in ds:
    if len(hle_examples) >= 20:
        break
    if sample["answer_type"] == "multipleChoice" and not sample["image"]:
        hle_examples.append({
            "question": sample["question"],
            "answer":   sample["answer"],
            "category": sample["category"],
        })

print(f"Loaded {len(hle_examples)} examples")
print("Categories:", set(e["category"] for e in hle_examples))

Loaded 20 examples
Categories: {'Computer Science/AI', 'Humanities/Social Science', 'Other', 'Math', 'Biology/Medicine'}


In [7]:
# Push to a LangSmith dataset
DATASET_NAME = "hle-mc-benchmark"

# Create dataset (skip if it already exists)
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description="HLE multiple-choice subset")
    client.create_examples(
        inputs           = [{"question": e["question"]} for e in hle_examples],
        outputs          = [{"answer":   e["answer"]}   for e in hle_examples],
        dataset_id       = dataset.id,
    )
    print(f"Created dataset with {len(hle_examples)} examples")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload")

Dataset 'hle-mc-benchmark' already exists — skipping upload


## 3. Run an experiment

A **target function** maps dataset inputs → outputs. An **evaluator** scores those outputs.

In [8]:
def target_gpt4o_mini(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are answering a multiple-choice exam question. Reply with only the letter of the correct answer (A, B, C, or D)."},
            {"role": "user",   "content": inputs["question"]},
        ],
        max_tokens=10,
    )
    return {"answer": response.choices[0].message.content.strip()}

In [9]:
# Evaluator: Anthropic judge for correctness
_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

In [10]:
results_mini = client.evaluate(
    target_gpt4o_mini,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini",
    max_concurrency=2,
)
print(results_mini)

View the evaluation results for experiment: 'gpt-4o-mini-a3788a1b' at:
https://eu.smith.langchain.com/o/87335986-1828-40f0-9dbd-323bf1524826/datasets/7d72e56b-14f4-4c09-bd15-2f66f6d0dd12/compare?selectedSessions=4c281ca4-2068-46bc-86c8-b38ca4a7ee87




19it [01:45,  7.11s/it]Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 019e6889-d0f8-7b32-aff0-91a49af339de: KeyError('score')
Traceback (most recent call last):
  File "/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/langsmith/evaluation/_runner.py", line 1596, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/langsmith/evaluation/evaluator.py", line 332, in evaluate_run
    result = self.func(
        run,
        example,
        langsmith_extra={"run_id": evaluator_run_id, "metadata": metadata},
    )
  File "/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/langsmith/evaluation/evaluator.py", line 758, in wrapper
    return func(*args, **kwargs)
  File "/tmp/ipykernel_238157/1795238459.py", line 9, in correctn

<ExperimentResults gpt-4o-mini-a3788a1b>


## 4. Compare two models (A/B)

Run the same benchmark with a different model. The LangSmith UI lets you compare both experiments side-by-side.

In [11]:
def target_gpt4o(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are answering a multiple-choice exam question. Reply with only the letter of the correct answer (A, B, C, or D)."},
            {"role": "user",   "content": inputs["question"]},
        ],
        max_tokens=10,
    )
    return {"answer": response.choices[0].message.content.strip()}

results_4o = client.evaluate(
    target_gpt4o,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o",
    max_concurrency=2,
)
print(results_4o)

View the evaluation results for experiment: 'gpt-4o-d6ccf9cc' at:
https://eu.smith.langchain.com/o/87335986-1828-40f0-9dbd-323bf1524826/datasets/7d72e56b-14f4-4c09-bd15-2f66f6d0dd12/compare?selectedSessions=41943530-8586-44bc-807a-bc89640c015e




20it [01:59,  5.99s/it]

<ExperimentResults gpt-4o-d6ccf9cc>


In [12]:
import pandas as pd

scores_mini = [r["feedback"]["correctness"] for r in results_mini._results if "correctness" in r.get("feedback", {})]
scores_4o   = [r["feedback"]["correctness"] for r in results_4o._results   if "correctness" in r.get("feedback", {})]

summary = pd.DataFrame({
    "model":     ["gpt-4o-mini", "gpt-4o"],
    "mean_score": [pd.Series(scores_mini).mean(), pd.Series(scores_4o).mean()],
    "n":         [len(scores_mini), len(scores_4o)],
})
print(summary)
print("\n→ Are these scores actually different? See notebook 03 for the statistical test.")

         model  mean_score  n
0  gpt-4o-mini         NaN  0
1       gpt-4o         NaN  0

→ Are these scores actually different? See notebook 03 for the statistical test.


## 5. Reproducibility

From the slides: same code + seed ≠ guaranteed identical results. Even small implementation differences can shift scores.

In [13]:
# Run the same single example twice — do you always get the same answer?
question = hle_examples[0]["question"]

run1 = target_gpt4o_mini({"question": question})
run2 = target_gpt4o_mini({"question": question})

print("Run 1:", run1["answer"])
print("Run 2:", run2["answer"])
print("Identical:", run1["answer"] == run2["answer"])
print("\nAt temperature=0 this is usually stable — but never guaranteed across API versions.")

Run 1: C
Run 2: C. Non-Elitism
Identical: False

At temperature=0 this is usually stable — but never guaranteed across API versions.
